# Llama 3.1 - Prompt Robustness Analysis for Social Media Dataset

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

In [ ]:
import json
from tqdm.auto import tqdm

In [ ]:
import sys
sys.path.append('jrcai_corekit')

In [ ]:
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [ ]:
from llms_corekit.llm import *

In [ ]:
from notebooks.Arabic_synthetic_dataset_generation.SocialMediaDatasetForPromptRobustness.prompts import PROMPTS
from notebooks.Arabic_synthetic_dataset_generation.SocialMediaDatasetForPromptRobustness.sample_posts import sample_and_save_posts

## Configuration

In [ ]:
BATCH_SIZE = 2
OUTPUT_BASE_DIR = "generated_arabic_datasets/llama-batched/arabic_social_media_dataset_prompt_robustness"

os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_BASE_DIR}")

## Initialize Model

In [ ]:
llm_loader = LLMLoader(
    '/hdd/shared_models/Meta-Llama-3.1-70B-Instruct/',
    llm_initializer=Llama31Initializer(),
)
message_generator = MessageGeneratorFromLocalLLM(llm_loader)

In [ ]:
def get_chat_completion(messages):
    llm_response = message_generator(messages)
    return llm_response[0]

In [ ]:
print(get_chat_completion(
    messages=[[{
        'role': 'user',
        'content': 'ما هو لون نجوم السماء؟'
    }]]
))

## Load and Sample Posts

In [ ]:
sampled_posts_file = os.path.join(OUTPUT_BASE_DIR, "sampled_posts.json")
sampled_posts, sampled_indices = sample_and_save_posts(
    posts_path='arabic_datasets/social_media_dataset.json',
    ouptut_path=sampled_posts_file,
)

## Utility Functions

In [ ]:
def save_posts_to_jsonl(posts, file_path):
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    with open(file_path, 'w', encoding='utf-8') as f:
        for post in posts:
            f.write(json.dumps(post, ensure_ascii=False) + '\n')


def load_existing_posts(file_path):
    if os.path.exists(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            return [json.loads(line) for line in f]
    return []


def clean_generated_post(post):
    post = post.strip().strip('<END>').strip().strip('<START>').strip()
    end_token_index = post.find('<END>')
    if end_token_index > 0:
        post = post[:end_token_index]
    return post.strip()

## Generation Function

In [ ]:
def generate_posts_with_prompt(posts, post_indices, prompt_name, prompt_template, batch_size=2):
    file_path = os.path.join(OUTPUT_BASE_DIR, f"{prompt_name}_posts_generation.jsonl")
    
    generated_posts = load_existing_posts(file_path)
    start_idx = len(generated_posts)
    
    if start_idx >= len(posts):
        print(f"All posts already generated for {prompt_name}")
        return generated_posts
    
    print(f"\nGenerating with {prompt_name}...")
    print(f"Resuming from index {start_idx}")
    
    generation_prompts = [prompt_template.format(post=post) for post in posts]
    
    batched_messages = [
        [[{'role': 'user', 'content': prompt}] for prompt in generation_prompts[i:i+batch_size]]
        for i in range(0, len(generation_prompts), batch_size)
    ]
    
    start_batch_idx = start_idx // batch_size
    
    for batch_idx, batch in enumerate(
        tqdm(batched_messages[start_batch_idx:], desc=f"Processing {prompt_name}"),
        start=start_batch_idx
    ):
        generated_batch = message_generator(batch)
        batch_generated_posts = [clean_generated_post(post) for post in generated_batch]
        
        batch_start = batch_idx * batch_size
        batch_end = min((batch_idx + 1) * batch_size, len(posts))
        
        batch_results = [
            {
                "original_post": post,
                "prompt_name": prompt_name,
                "generated_post": generated_post,
                "original_index": post_indices[batch_start + i]
            }
            for i, (post, generated_post) in enumerate(
                zip(posts[batch_start:batch_end], batch_generated_posts)
            )
        ]
        
        generated_posts.extend(batch_results)
        save_posts_to_jsonl(generated_posts, file_path)
    
    print(f"Completed {prompt_name}: {len(generated_posts)} posts saved to {file_path}")
    return generated_posts

## Generate Posts with All 5 Prompts

In [ ]:
print(f"Loaded {len(PROMPTS)} prompts:")
for prompt_name in PROMPTS.keys():
    print(f"  - {prompt_name}")

In [ ]:
results = {}

for prompt_name, prompt_template in PROMPTS.items():
    results[prompt_name] = generate_posts_with_prompt(
        posts=sampled_posts,
        post_indices=sampled_indices,
        prompt_name=prompt_name,
        prompt_template=prompt_template,
        batch_size=BATCH_SIZE
    )

print("\n" + "="*50)
print("All prompts completed!")
print("="*50)

## Summary

In [ ]:
print("\nGeneration Summary:")
print(f"Number of sampled posts: {len(sampled_posts)}")
print(f"Number of prompts: {len(PROMPTS)}")
print(f"Batch size: {BATCH_SIZE}")
print(f"\nOutput directory: {OUTPUT_BASE_DIR}")
print(f"\nGenerated files:")
for prompt_name in PROMPTS.keys():
    file_path = os.path.join(OUTPUT_BASE_DIR, f"{prompt_name}_posts_generation.jsonl")
    if os.path.exists(file_path):
        num_lines = len(load_existing_posts(file_path))
        print(f"  - {prompt_name}: {num_lines} posts")